# LSTM-IDS — Multi-Dataset Pipeline (Master Notebook)

Runs **all 3 datasets** sequentially: NSL-KDD → UNSW-NB15 → CICIDS2017.

Each dataset produces its own output directory under `outputs/<dataset>/`.
All figures are saved per-dataset — no cross-contamination.

**Workflow:**
1. Mount + Git pull + Install (Cells 1–4)
2. For each dataset: Extract → Run stages 1–9
3. Backup to Drive

**Resume:** If interrupted, re-run the failed cell. Completed stages are
skipped via `--resume`.

**Runtime:** ~2–4 hours per dataset on CPU. Use GPU runtime if available.

Use `Runtime > Restart runtime` after git pull if you hit import issues.

---
## Setup (run once)

### Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('Google Drive mounted.')

### Cell 2 — Clone / Pull from GitHub

In [ ]:
import os, subprocess

PROJECT_DIR = '/content/lstm_ids_project'
REPO_URL = 'https://github.com/Nico3783/lstm_ids_project.git'

try:
    from google.colab import userdata
    GH_TOKEN = userdata.get('GH_TOKEN')
    AUTH_URL = REPO_URL.replace('https://', f'https://{GH_TOKEN}@') if GH_TOKEN else REPO_URL
except Exception:
    AUTH_URL = REPO_URL

if os.path.exists(PROJECT_DIR):
    print('Pulling latest...')
    r = subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--ff-only'], capture_output=True, text=True)
    if r.returncode != 0:
        subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', 'origin'], check=True)
        subprocess.run(['git', '-C', PROJECT_DIR, 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', AUTH_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
!git log --oneline -3

### Cell 3 — Install Dependencies

In [ ]:
import os

# Rename local TF shim so real TF is used
if os.path.isdir('tensorflow') and not os.path.isdir('tensorflow_shim_local'):
    os.rename('tensorflow', 'tensorflow_shim_local')
    print('Renamed tensorflow/ -> tensorflow_shim_local/')

!pip install -q -r requirements_colab.txt
print('Dependencies installed.')

### Cell 4 — Environment Validation

In [ ]:
import os, platform, sys

PROJECT = '/content/lstm_ids_project'
os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
os.environ['KERAS_BACKEND'] = 'tensorflow'
os.environ['PYTHONPATH'] = PROJECT

# GPU
try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print(f'GPU: {gpus[0].name}')
        tf.config.experimental.set_memory_growth(gpus[0], True)
    else:
        print('WARNING: No GPU — all training will be CPU-only.')
except Exception as e:
    print(f'TF check failed: {e}')

# RAM
try:
    import psutil
    print(f'RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB')
except ImportError:
    pass

# Disk
st = os.statvfs('.')
print(f'Disk free: {(st.f_bavail * st.f_frsize) / 1024**3:.1f} GB')

# Import check
from src.pipeline.checkpoint import CheckpointManager
print(f'Pipeline import OK — {len(CheckpointManager.STAGES)} stages')

# Keepalive helper
import threading, time
def start_keepalive():
    def _hb():
        while True:
            print(f'[{time.strftime("%H:%M:%S")}] keepalive', flush=True)
            time.sleep(300)
    threading.Thread(target=_hb, daemon=True).start()
    print('Keepalive thread started.')

---
## NSL-KDD

### Cell 5 — Extract NSL-KDD Data

In [ ]:
import os

DATA_DIR = 'data/raw/nsl_kdd'
if os.path.isdir(DATA_DIR) and len(os.listdir(DATA_DIR)) >= 3:
    print('NSL-KDD data already exists — skipping extraction.')
else:
    import zipfile
    candidates = [
        '/content/drive/MyDrive/lstm_raw.zip',
        '/content/drive/MyDrive/lstm_ids_project/data/raw/lstm_raw.zip',
        '/content/drive/MyDrive/nsl_kdd.zip',
    ]
    zip_path = next((p for p in candidates if os.path.exists(p)), None)
    if zip_path:
        print(f'Extracting {zip_path}...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall('data/raw')
        print('Done.')
    else:
        print('WARNING: No NSL-KDD zip found on Drive.')

if os.path.isdir(DATA_DIR):
    files = os.listdir(DATA_DIR)
    print(f'NSL-KDD dir: {len(files)} files')
    for f in sorted(files):
        print(f'  {f}')
else:
    print('ERROR: data/raw/nsl_kdd/ missing.')

### Cell 6 — NSL-KDD: Stages 1–3 (Preprocessing → Split → Scale)

In [ ]:
import os
os.chdir('/content/lstm_ids_project')

!PYTHONPATH=/content/lstm_ids_project python run_pipeline.py \
    --dataset nsl_kdd --resume \
    --start-stage preprocessing --end-stage scale_sequences \
    --log-level INFO

### Cell 7 — NSL-KDD: Stage 5 — Baselines

In [ ]:
import os
os.chdir('/content/lstm_ids_project')
start_keepalive()

!PYTHONPATH=/content/lstm_ids_project python run_pipeline.py \
    --dataset nsl_kdd --resume \
    --start-stage baselines --end-stage baselines \
    --log-level INFO

### Cell 8 — NSL-KDD: Stage 6 — LSTM Training (~20–40 min)

In [ ]:
import os
os.chdir('/content/lstm_ids_project')
start_keepalive()

!PYTHONPATH=/content/lstm_ids_project python run_pipeline.py \
    --dataset nsl_kdd --resume \
    --start-stage lstm_train --end-stage lstm_train \
    --epochs 100 --batch-size 64 --log-level INFO

### Cell 9 — NSL-KDD: Stages 7–9 (Eval → Viz → Export)

In [ ]:
import os
os.chdir('/content/lstm_ids_project')

!PYTHONPATH=/content/lstm_ids_project python run_pipeline.py \
    --dataset nsl_kdd --resume \
    --start-stage evaluation --end-stage export \
    --zip-drive --log-level INFO

### Cell 10 — NSL-KDD: Backup to Drive

In [ ]:
import os, shutil

DRIVE_BACKUP = '/content/drive/MyDrive/lstm_ids_results/nsl_kdd'
os.makedirs(DRIVE_BACKUP, exist_ok=True)

LOCAL_OUT = 'outputs/nsl_kdd'
if os.path.isdir(LOCAL_OUT):
    for root, dirs, files in os.walk(LOCAL_OUT):
        for f in files:
            src = os.path.join(root, f)
            rel = os.path.relpath(src, LOCAL_OUT)
            dst = os.path.join(DRIVE_BACKUP, rel)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
    print(f'NSL-KDD backed up to {DRIVE_BACKUP}')
    # List figures
    fig_dir = os.path.join(LOCAL_OUT, 'reports', 'figures')
    if os.path.isdir(fig_dir):
        print(f'Figures ({len(os.listdir(fig_dir))}):')
        for f in sorted(os.listdir(fig_dir)):
            print(f'  {f}')
else:
    print('WARNING: outputs/nsl_kdd/ not found.')

---
## UNSW-NB15

### Cell 11 — Extract UNSW-NB15 Data

In [ ]:
import os

DATA_DIR = 'data/raw/unsw_nb15'
if os.path.isdir(DATA_DIR) and len(os.listdir(DATA_DIR)) >= 3:
    print('UNSW-NB15 data already exists — skipping extraction.')
else:
    import zipfile
    candidates = [
        '/content/drive/MyDrive/lstm_raw.zip',
        '/content/drive/MyDrive/lstm_ids_project/data/raw/lstm_raw.zip',
        '/content/drive/MyDrive/unsw_nb15.zip',
    ]
    zip_path = next((p for p in candidates if os.path.exists(p)), None)
    if zip_path:
        print(f'Extracting {zip_path}...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall('data/raw')
        print('Done.')
    else:
        print('WARNING: No UNSW-NB15 zip found on Drive.')

if os.path.isdir(DATA_DIR):
    files = os.listdir(DATA_DIR)
    print(f'UNSW-NB15 dir: {len(files)} files')
    for f in sorted(files):
        print(f'  {f}')
else:
    print('ERROR: data/raw/unsw_nb15/ missing.')

### Cell 12 — UNSW-NB15: Stages 1–3

In [ ]:
import os
os.chdir('/content/lstm_ids_project')

!PYTHONPATH=/content/lstm_ids_project python run_pipeline.py \
    --dataset unsw_nb15 --resume \
    --start-stage preprocessing --end-stage scale_sequences \
    --log-level INFO

### Cell 13 — UNSW-NB15: Stage 5 — Baselines

In [ ]:
import os
os.chdir('/content/lstm_ids_project')
start_keepalive()

!PYTHONPATH=/content/lstm_ids_project python run_pipeline.py \
    --dataset unsw_nb15 --resume \
    --start-stage baselines --end-stage baselines \
    --log-level INFO

### Cell 14 — UNSW-NB15: Stage 6 — LSTM Training

In [ ]:
import os
os.chdir('/content/lstm_ids_project')
start_keepalive()

!PYTHONPATH=/content/lstm_ids_project python run_pipeline.py \
    --dataset unsw_nb15 --resume \
    --start-stage lstm_train --end-stage lstm_train \
    --epochs 100 --batch-size 64 --log-level INFO

### Cell 15 — UNSW-NB15: Stages 7–9

In [ ]:
import os
os.chdir('/content/lstm_ids_project')

!PYTHONPATH=/content/lstm_ids_project python run_pipeline.py \
    --dataset unsw_nb15 --resume \
    --start-stage evaluation --end-stage export \
    --zip-drive --log-level INFO

### Cell 16 — UNSW-NB15: Backup to Drive

In [ ]:
import os, shutil

DRIVE_BACKUP = '/content/drive/MyDrive/lstm_ids_results/unsw_nb15'
os.makedirs(DRIVE_BACKUP, exist_ok=True)

LOCAL_OUT = 'outputs/unsw_nb15'
if os.path.isdir(LOCAL_OUT):
    for root, dirs, files in os.walk(LOCAL_OUT):
        for f in files:
            src = os.path.join(root, f)
            rel = os.path.relpath(src, LOCAL_OUT)
            dst = os.path.join(DRIVE_BACKUP, rel)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
    print(f'UNSW-NB15 backed up to {DRIVE_BACKUP}')
    fig_dir = os.path.join(LOCAL_OUT, 'reports', 'figures')
    if os.path.isdir(fig_dir):
        print(f'Figures ({len(os.listdir(fig_dir))}):')
        for f in sorted(os.listdir(fig_dir)):
            print(f'  {f}')
else:
    print('WARNING: outputs/unsw_nb15/ not found.')

---
## CICIDS2017 (optional — may skip for now)

### Cell 17 — Extract CICIDS2017 Data

In [ ]:
import os

DATA_DIR = 'data/raw/cicids2017'
if os.path.isdir(DATA_DIR) and len(os.listdir(DATA_DIR)) >= 3:
    print('CICIDS2017 data already exists — skipping extraction.')
else:
    import zipfile
    candidates = [
        '/content/drive/MyDrive/lstm_raw.zip',
        '/content/drive/MyDrive/lstm_ids_project/data/raw/lstm_raw.zip',
        '/content/drive/MyDrive/cicids2017.zip',
    ]
    zip_path = next((p for p in candidates if os.path.exists(p)), None)
    if zip_path:
        print(f'Extracting {zip_path}...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall('data/raw')
        print('Done.')
    else:
        print('WARNING: No CICIDS2017 zip found on Drive.')

if os.path.isdir(DATA_DIR):
    files = os.listdir(DATA_DIR)
    print(f'CICIDS2017 dir: {len(files)} files')
    for f in sorted(files)[:10]:
        print(f'  {f}')
    if len(files) > 10:
        print(f'  ... and {len(files) - 10} more')
else:
    print('ERROR: data/raw/cicids2017/ missing.')

### Cell 18 — CICIDS2017: Stages 1–3

In [ ]:
import os
os.chdir('/content/lstm_ids_project')

!PYTHONPATH=/content/lstm_ids_project python run_pipeline.py \
    --dataset cicids2017 --resume \
    --start-stage preprocessing --end-stage scale_sequences \
    --log-level INFO

### Cell 19 — CICIDS2017: Stage 5 — Baselines

In [ ]:
import os
os.chdir('/content/lstm_ids_project')
start_keepalive()

!PYTHONPATH=/content/lstm_ids_project python run_pipeline.py \
    --dataset cicids2017 --resume \
    --start-stage baselines --end-stage baselines \
    --log-level INFO

### Cell 20 — CICIDS2017: Stage 6 — LSTM Training

In [ ]:
import os
os.chdir('/content/lstm_ids_project')
start_keepalive()

!PYTHONPATH=/content/lstm_ids_project python run_pipeline.py \
    --dataset cicids2017 --resume \
    --start-stage lstm_train --end-stage lstm_train \
    --epochs 100 --batch-size 64 --log-level INFO

### Cell 21 — CICIDS2017: Stages 7–9

In [ ]:
import os
os.chdir('/content/lstm_ids_project')

!PYTHONPATH=/content/lstm_ids_project python run_pipeline.py \
    --dataset cicids2017 --resume \
    --start-stage evaluation --end-stage export \
    --zip-drive --log-level INFO

### Cell 22 — CICIDS2017: Backup to Drive

In [ ]:
import os, shutil

DRIVE_BACKUP = '/content/drive/MyDrive/lstm_ids_results/cicids2017'
os.makedirs(DRIVE_BACKUP, exist_ok=True)

LOCAL_OUT = 'outputs/cicids2017'
if os.path.isdir(LOCAL_OUT):
    for root, dirs, files in os.walk(LOCAL_OUT):
        for f in files:
            src = os.path.join(root, f)
            rel = os.path.relpath(src, LOCAL_OUT)
            dst = os.path.join(DRIVE_BACKUP, rel)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
    print(f'CICIDS2017 backed up to {DRIVE_BACKUP}')
    fig_dir = os.path.join(LOCAL_OUT, 'reports', 'figures')
    if os.path.isdir(fig_dir):
        print(f'Figures ({len(os.listdir(fig_dir))}):')
        for f in sorted(os.listdir(fig_dir)):
            print(f'  {f}')
else:
    print('WARNING: outputs/cicids2017/ not found.')

---
## Summary

### Cell 23 — Verify All Outputs

In [ ]:
import os

for dataset in ['nsl_kdd', 'unsw_nb15', 'cicids2017']:
    out = f'outputs/{dataset}'
    fig_dir = os.path.join(out, 'reports', 'figures')
    tables_dir = os.path.join(out, 'reports', 'tables')
    models_dir = os.path.join(out, 'models', 'final')
    print(f'\n=== {dataset.upper()} ===')
    if os.path.isdir(fig_dir):
        figs = sorted(os.listdir(fig_dir))
        print(f'  Figures ({len(figs)}): {figs}')
    else:
        print(f'  No figures directory')
    if os.path.isdir(tables_dir):
        tabs = sorted(os.listdir(tables_dir))
        print(f'  Tables ({len(tabs)}): {tabs}')
    if os.path.isdir(models_dir):
        mods = sorted(os.listdir(models_dir))
        print(f'  Models ({len(mods)}): {mods}')

# Drive backup verification
print('\n=== DRIVE BACKUPS ===')
for dataset in ['nsl_kdd', 'unsw_nb15', 'cicids2017']:
    drive_dir = f'/content/drive/MyDrive/lstm_ids_results/{dataset}'
    if os.path.isdir(drive_dir):
        count = sum(len(files) for _, _, files in os.walk(drive_dir))
        print(f'  {dataset}: {count} files backed up')
    else:
        print(f'  {dataset}: no backup found')